<a href="https://colab.research.google.com/github/EthanIsaac/valorant-voiceline-qa/blob/main/notebooks/04_spanish_subtitles.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04 · Spanish Subtitles: comparing 3 pre-trained translation models

**Project:** VALORANT Voice-Line QA & Localization Assistant (Riot Games)
**Pipeline:** `translation` (English → Spanish), which drafts a Spanish subtitle for each voice line, for Riot's localization team to review.

**Why it matters:** every voice line is localized into many languages. A good first draft saves translator time; a bad one (dropped sentences, translated agent names, lost callouts) costs more than it saves.

**How the models are scored.** The dataset has no Spanish reference subtitles, so there is no automatic "correct answer" to compare against. Two measures are used instead:
1. **Human rating (1–5)** of a blind sample of 30 lines by a Spanish speaker: the main measure.
2. **Names and callouts preserved** (automatic, all test lines): agent names (*Jett*), site callouts (*A*, *B*, *C*, *Mid*) must survive translation unchanged, or the subtitle is wrong in a way players notice immediately.

**Notebook outline**
- Step 0: Setup (packages, seed, output folder)
- Step 1: Load the English test lines (same split as notebooks 01–03)
- Step 2: Choose the 3 candidate models
- Step 3: Translate the test lines
- Step 4: Check that agent names and callouts are preserved
- Step 5: Measure CPU runtime through a `pipeline` (Streamlit Cloud runs on CPU)
- Step 6: Create the blind rating sheet
- Step 7: Read the human ratings, compare the models and pick the best
- Save and download the results (with SHA-256 fingerprints)

All results are written to `results/translation/`.

> Runtime: **GPU** (Runtime → Change runtime type → T4 GPU). The GPU only speeds up Step 3; runtime is measured on CPU in Step 5.

# Step 0: Setup

In [ ]:
# Same transformers constraint as the course notebooks; sentencepiece and sacremoses are needed by the translation tokenizers.
# Expected: a pip "dependency conflicts" message about diffusers/gradio needing a newer
# huggingface-hub. Those packages are not used here, so it can be ignored.
%pip install -q "transformers>=4.46.0,<5.0.0" accelerate sentencepiece sacremoses

In [ ]:
import hashlib
import io
import json
import os
import re
import time
import urllib.error
import urllib.request

import numpy as np
import pandas as pd
import torch
import transformers
from IPython.display import display
from transformers import set_seed

# One seed for everything; translation itself is deterministic (beam search, no sampling)
SEED = 42
set_seed(SEED)

# Every result file produced by this notebook goes in this folder
RESULTS_DIR = "results/translation"
os.makedirs(RESULTS_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"transformers {transformers.__version__} | torch {torch.__version__} | device: {device}")
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# Step 1: Load the English test lines

The **same split as notebooks 01–03** (`results/agent_classifier/splits.csv`), downloaded from GitHub at a fixed commit and verified with its SHA-256 fingerprint.

The source text is the **script line** (`spoken_text`), not a speech-to-text transcript, so that translation quality is measured on its own, without speech-recognition errors mixed in.
Only `english_only` lines are used, and each distinct line is translated once (shared callouts such as *"Let's rush A."* appear for several agents).
No audio is needed in this notebook.

In [ ]:
REPO_COMMIT = "2837dcb342749a346d4370e63545c575ba5c2a07"   # commit with the notebook 01 results
SPLITS_URL = f"https://raw.githubusercontent.com/EthanIsaac/valorant-voiceline-qa/{REPO_COMMIT}/results/agent_classifier/splits.csv"
SPLITS_SHA256 = "72b658b97258de7bc3e77a2b88908080dd46be76d0162c8905cceff2c903dd7d"   # from results/agent_classifier/SHA256SUMS

splits_bytes = urllib.request.urlopen(SPLITS_URL).read()
assert hashlib.sha256(splits_bytes).hexdigest() == SPLITS_SHA256, "splits.csv does not match notebook 01"
splits = pd.read_csv(io.BytesIO(splits_bytes))

test_lines = splits.loc[(splits["split"] == "test") & splits["english_only"], "spoken_text"]
lines = pd.DataFrame({"source": sorted(test_lines.unique())})
lines.insert(0, "line_id", range(len(lines)))
print(f"{len(test_lines)} English test clips -> {len(lines)} distinct lines")
print(f"Words per line: median {lines['source'].str.split().str.len().median():.0f}, max {lines['source'].str.split().str.len().max()}")
lines.sample(5, random_state=SEED)

# Step 2: Choose the 3 candidate models

| Key | Hugging Face model | Size | Why it's a candidate |
|---|---|---|---|
| `opus_mt` | `Helsinki-NLP/opus-mt-en-es` | ~78M | Small model trained for English → Spanish only: the fastest option |
| `m2m100` | `facebook/m2m100_418M` | ~484M | Multilingual (100 languages): one model could cover every language Riot localizes into |
| `nllb` | `facebook/nllb-200-distilled-600M` | ~615M | Meta's newer multilingual model (200 languages), distilled from a larger one; strong on short, informal text |

**Why these three:** they compare a small **single-pair** model with two larger **multilingual** models, which a localization team supporting many languages would prefer if quality allowed.
All three are standard, freely available translation models on the Hugging Face Hub; none is a large language model.

**Same decoding for all:** beam search with 4 beams and at most 128 new tokens. (The models ship with different defaults, from 1 to 5 beams; using the same setting keeps the comparison fair.)

In [ ]:
from transformers import pipeline

CANDIDATES = {
    "opus_mt": {"model": "Helsinki-NLP/opus-mt-en-es"},
    "m2m100": {"model": "facebook/m2m100_418M", "src_lang": "en", "tgt_lang": "es"},
    "nllb": {"model": "facebook/nllb-200-distilled-600M", "src_lang": "eng_Latn", "tgt_lang": "spa_Latn"},
}
GENERATION = {"num_beams": 4, "max_new_tokens": 128}


def load_translator(key, device_index):
    return pipeline("translation", device=device_index, **CANDIDATES[key])


def translate(translator, texts):
    """One line at a time, like the app."""
    return [translator(t, **GENERATION)[0]["translation_text"].strip() for t in texts]

# Step 3: Translate the test lines

Each model translates every distinct test line, one line at a time (as in the app). The GPU is used here only to make this step faster.

In [ ]:
translations = lines.copy()
gpu_seconds = {}

for key in CANDIDATES:
    translator = load_translator(key, 0 if device == "cuda" else -1)
    start = time.perf_counter()
    translations[key] = translate(translator, lines["source"])
    gpu_seconds[key] = round(time.perf_counter() - start, 1)
    print(f"{key:<8} {len(lines)} lines in {gpu_seconds[key]:.0f}s")
    del translator
    torch.cuda.empty_cache()

translations.sample(10, random_state=SEED)

# Step 4: Check that agent names and callouts are preserved

A subtitle must keep **agent names** and **map callouts** exactly as in the game: they are proper names, not words to translate.
For every line that contains one, the check asks whether each of them appears in the translation:
- agent names (*Jett*, *KAY/O*, …) and *Mid*: present, ignoring case
- site letters (*A*, *B*, *C* as a separate word): present **in capitals**, since in Spanish a lower-case *a* is a preposition (in the English line, *A* followed by an ordinary word, as in *"A little help."*, is the article, not a site)

This also catches **dropped sentences**, a common failure of translation models on multi-sentence lines (e.g. *"Jett, watch my back. Sova, scan Mid."* translated without the second sentence).

In [ ]:
AGENT_NAMES = ["Astra", "Breach", "Brimstone", "Chamber", "Clove", "Cypher", "Deadlock", "Fade", "Gekko",
               "Harbor", "Iso", "Jett", "KAY/O", "Killjoy", "Neon", "Omen", "Phoenix", "Raze", "Reyna",
               "Sage", "Skye", "Sova", "Tejo", "Viper", "Vyse", "Yoru"]
NAME_PATTERNS = {name: re.compile(rf"(?<!\w){re.escape(name)}(?!\w)", re.IGNORECASE) for name in AGENT_NAMES + ["Mid"]}
SITE_PATTERNS = {site: re.compile(rf"(?<!\w){site}(?!\w)") for site in ["A", "B", "C"]}   # case-sensitive
# In English, "A" is also an article ("A little help.") or part of a word ("A-okay"): in the source it only
# counts as a site when it is not followed by an ordinary lower-case word (callout words such as "long" excepted)
SOURCE_SITE_PATTERNS = {**SITE_PATTERNS, "A": re.compile(
    r"(?<!\w)A(?![\w-])(?! (?!(?:site|long|short|main|link|lobby|heaven)\b)[a-z])")}


def protected_terms(text):
    """Agent names, Mid and site letters that appear in an English line."""
    terms = [name for name, p in NAME_PATTERNS.items() if p.search(text)]
    return terms + [site for site, p in SOURCE_SITE_PATTERNS.items() if p.search(text)]


def preserved(term, translation):
    return bool({**NAME_PATTERNS, **SITE_PATTERNS}[term].search(translation))


translations["protected_terms"] = translations["source"].map(protected_terms)
has_terms = translations["protected_terms"].str.len() > 0
print(f"{has_terms.sum()} of {len(translations)} lines contain an agent name or callout")

preservation = {}
for key in CANDIDATES:
    kept = translations.loc[has_terms].apply(lambda r: all(preserved(t, r[key]) for t in r["protected_terms"]), axis=1)
    translations.loc[has_terms, f"{key}_terms_preserved"] = kept
    preservation[key] = float(kept.mean())
    print(f"{key:<8} all names/callouts preserved in {kept.mean():.1%} of those lines")

translations.to_csv(f"{RESULTS_DIR}/test_translations.csv", index=False)   # now with the check per line

# Examples of lines where a model lost a name or callout
lost = translations[has_terms & ~translations[[f"{k}_terms_preserved" for k in CANDIDATES]].fillna(True).all(axis=1)]
lost[["source"] + list(CANDIDATES)].head(10)

# Step 5: Measure CPU runtime through a `pipeline`

The deployed app runs on **Streamlit Cloud, which has no GPU**, so runtime is measured on CPU with the same `pipeline` call the app will use, following the course method:
- **with model loading**: `cpu_load_s` (model files downloaded first, so network speed is not counted)
- **without model loading**: `cpu_ms_per_line`, over the first 100 test lines

As a consistency check, the CPU translations must match Step 3 (`cpu_agreement`).

In [ ]:
from huggingface_hub import snapshot_download

speed_lines = lines["source"].head(100)
results = {}

for key, config in CANDIDATES.items():
    snapshot_download(config["model"])   # make sure the files are cached, so only loading is timed

    start = time.perf_counter()
    translator = load_translator(key, -1)   # -1 means CPU
    load_s = time.perf_counter() - start
    translate(translator, speed_lines.head(1))   # warm-up

    start = time.perf_counter()
    cpu_out = translate(translator, speed_lines)
    ms_per_line = (time.perf_counter() - start) / len(speed_lines) * 1000

    results[key] = {
        "model_id": config["model"],
        "params_millions": round(translator.model.num_parameters() / 1e6, 1),
        "terms_preserved": preservation[key],
        "gpu_seconds_test_lines": gpu_seconds[key],
        "cpu_load_s": round(load_s, 2),
        "cpu_ms_per_line": round(ms_per_line, 1),
        "cpu_agreement": float(np.mean([a == b for a, b in zip(cpu_out, translations[key].head(100))])),
    }
    print(f"{key:<8} load={load_s:5.2f}s  per line={ms_per_line:6.1f}ms  agreement with Step 3={results[key]['cpu_agreement']:.0%}")
    del translator

pd.DataFrame(results).T

# Step 6: Create the blind rating sheet

**30 lines** are drawn at random (seeded) from the test lines. For each line the three translations are shown in a random order as options **A, B, C**, so the rater does not know which model produced which.

**Rating scale** (one score per translation):

| Score | Meaning |
|---|---|
| 5 | Correct and natural; usable as a subtitle as is |
| 4 | Correct meaning; minor style or wording issues |
| 3 | Understandable, but with an error a translator must fix |
| 2 | Meaning partly wrong, or part of the line missing |
| 1 | Wrong, untranslated or unusable |

Agent names and callouts translated or lost count as errors.

**Workflow:** the sheet `rating_sheet.csv` is downloaded with the results. The rater fills in the `score` column and the file is committed to the repository as `results/translation/human_ratings.csv`, where Step 7 reads it.

In [ ]:
RATED_LINES = 30
rng = np.random.default_rng(SEED)
sample = translations.loc[np.sort(rng.choice(len(translations), RATED_LINES, replace=False))]

sheet_rows, key_rows = [], []
for _, row in sample.iterrows():
    order = rng.permutation(list(CANDIDATES))   # random order of the models for this line
    for option, model_key in zip("ABC", order):
        sheet_rows.append({"line_id": row["line_id"], "source": row["source"], "option": option,
                           "translation": row[model_key], "score": ""})
        key_rows.append({"line_id": row["line_id"], "option": option, "model": model_key})

rating_sheet = pd.DataFrame(sheet_rows)
rating_key = pd.DataFrame(key_rows)   # kept in memory only, so the sheet stays blind
rating_sheet.to_csv(f"{RESULTS_DIR}/rating_sheet.csv", index=False)
rating_sheet.head(9)

# Step 7: Read the human ratings, compare the models and pick the best

The completed sheet is read from the repository. Before using it, every translation in it must match this run's translations exactly; otherwise the ratings would not belong to these outputs.

**Selection rule** (decided before rating):
1. Highest **mean human score**.
2. If several models are within 0.25 points of the best, choose the **fastest on CPU**, because the app runs on CPU.

If the ratings file is not in the repository yet, this step is skipped.

In [ ]:
RATINGS_URL = "https://raw.githubusercontent.com/EthanIsaac/valorant-voiceline-qa/main/results/translation/human_ratings.csv"

try:
    ratings = pd.read_csv(io.BytesIO(urllib.request.urlopen(RATINGS_URL).read()))
except urllib.error.HTTPError:
    ratings = None
    print("No human ratings in the repository yet: fill in rating_sheet.csv and commit it as results/translation/human_ratings.csv.")

SELECTED = None
if ratings is not None:
    merged = ratings.merge(rating_sheet[["line_id", "option", "translation"]], on=["line_id", "option"], suffixes=("", "_run"))
    assert len(merged) == len(rating_sheet), "the ratings do not cover this run's rating sheet"
    assert (merged["translation"] == merged["translation_run"]).all(), "the rated translations differ from this run's"
    assert merged["score"].between(1, 5).all(), "every score must be 1-5"

    scored = merged.merge(rating_key, on=["line_id", "option"])
    scored[["line_id", "source", "option", "model", "translation", "score"]].to_csv(
        f"{RESULTS_DIR}/human_ratings_unblinded.csv", index=False)

    for key in CANDIDATES:
        scores = scored.loc[scored["model"] == key, "score"]
        results[key].update({
            "human_score_mean": scores.mean(),
            "human_score_std": scores.std(),
            "share_usable_4_or_5": float((scores >= 4).mean()),
        })

    comparison = pd.DataFrame(results).T.infer_objects().rename_axis("model").reset_index()
    best = comparison["human_score_mean"].max()
    close = comparison[comparison["human_score_mean"] >= best - 0.25]
    SELECTED = close.sort_values("cpu_ms_per_line").iloc[0]["model"]
    comparison["selected"] = comparison["model"] == SELECTED
    comparison.to_csv(f"{RESULTS_DIR}/model_comparison.csv", index=False)
    print("Selected:", SELECTED, "->", CANDIDATES[SELECTED]["model"])
    display(comparison)
    display(scored.pivot_table(index="score", columns="model", values="line_id", aggfunc="count", fill_value=0))

In [ ]:
# Record exactly how these results were produced
with open(f"{RESULTS_DIR}/run_config.json", "w") as f:
    json.dump({
        "seed": SEED,
        "splits_csv": {"url": SPLITS_URL, "sha256": SPLITS_SHA256},
        "test_lines": len(lines),
        "candidates": CANDIDATES,
        "generation": GENERATION,
        "rated_lines": RATED_LINES,
        "ratings_url": RATINGS_URL,
        "ratings_found": ratings is not None,
        "selection_rule": "highest mean human score; ties within 0.25 go to lowest cpu_ms_per_line",
        "selected_model": SELECTED,
        "versions": {"transformers": transformers.__version__, "torch": torch.__version__},
        "gpu": torch.cuda.get_device_name(0) if device == "cuda" else None,
        "run_at_utc": pd.Timestamp.now(tz="UTC").isoformat(timespec="seconds"),
    }, f, indent=2)
print(open(f"{RESULTS_DIR}/run_config.json").read())

# Save and download the results

Every file this notebook generated is listed with its **SHA-256 fingerprint** in `SHA256SUMS`.
Translations are deterministic (beam search), so they should be identical on a rerun; the runtime figures differ slightly.
Then everything is zipped and downloaded as `results_translation.zip`.

In [ ]:
from pathlib import Path

result_files = sorted(p for p in Path(RESULTS_DIR).rglob("*") if p.is_file() and p.name != "SHA256SUMS")
with open(f"{RESULTS_DIR}/SHA256SUMS", "w") as f:
    for p in result_files:
        digest = hashlib.sha256(p.read_bytes()).hexdigest()
        f.write(f"{digest}  {p.relative_to(RESULTS_DIR)}\n")
        print(f"{digest[:16]}…  {p.relative_to(RESULTS_DIR)}  ({p.stat().st_size:,} bytes)")

In [ ]:
!rm -f results_translation.zip && zip -qrX results_translation.zip results/translation
from google.colab import files
files.download("results_translation.zip")